In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

**Preprocessing Data**

In [2]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [3]:
df = pd.read_json('datasets/news.json').drop('source',axis=1)

In [4]:
df.head()

,article,orientation
0,Health authorities in one state have issued an...,western_conservative
1,\n'Kennedy Saves the World' podcast host Kenne...,western_conservative
2,\nFormer counterterrorism analyst Jonathan Sch...,western_conservative
3,\nFox News Flash top headlines are here. Check...,western_conservative
4,\nCrowe is charged with harassment and stalkin...,western_conservative


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1164 entries, 0 to 1163
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   article      1164 non-null   object
 1   orientation  1164 non-null   object
dtypes: object(2)
memory usage: 18.3+ KB


In [6]:
df['orientation'].unique()

array(['western_conservative', 'non_western', 'western_progressive'],
      dtype=object)

In [7]:
# Turn categories into numbers

# Define the categorical features
categorical_features = ['orientation']

# Initialize the OneHotEncoder
one_hot = OneHotEncoder()

# Initialize the ColumnTransformer
transformer = ColumnTransformer([('one_hot', one_hot, categorical_features)], remainder='passthrough')

# Apply the transformation to your dataframe
df_transformed = transformer.fit_transform(df)
df_transformed[:3]

array([[0.0, 1.0, 0.0,
        'Health authorities in one state have issued an urgent alert for residents who visited a Costco, DFO, businesses and caught trams after two measles cases were infectious while in public.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nVictorian residents have been put on alert after two holidaymakers returning from overseas were unknowingly infectious with measles while out in the community.\nThe Department of Health revealed the new cases on Saturday afternoon, which brings the total measles cases to three after another traveller was identified this week.\nAt least 10 exposure sites have been listed, with the days ranging between Wednesday January 17 and Wednesday January 24, on the department\'s website.\nWant more news? Stream Sky News Australia’s live channel here. \nWednesday January 17 \n6am to 3pm: Bay City Auto Group (and associated construction site) 14 Dandenong Road West, Frankston\n7:30pm to 9pm: Box Hill Action Indoor Sports 9 Clarice Road, Box Hill

In [8]:
# Turning it into
data = pd.DataFrame(df_transformed)
data.columns = ['western_conservative','non_western','western_progressive','article']

# Removing \n and pre-word-embedding cleaning
char = '\\'
data['article'] = data['article'].str.replace('"','')
data['article'] = data['article'].str.replace("'","")
data['article'] = data['article'].str.replace(',','')
data['article'] = data['article'].str.replace('.','')
data['article'] = data['article'].str.lower()
data['article'] = data['article'].str.replace('\n','')
data['article'] = data['article'].str.replace(char,'')
data['article'] = data['article'].str.replace('/','')
data['article'] = data['article'].str.replace('—','')
data['article'] = data['article'].str.replace('_','')
data['article'] = data['article'].str.replace('’','')
data['article'] = data['article'].str.replace('-','')
data['article'] = data['article'].str.replace('@','')
data['article'] = data['article'].str.replace('–','')
data['article'] = data['article'].str.replace('‘','')
data['article'] = data['article'].str.replace('…','')
data['article'] = data['article'].str.replace('”','')
data['article'] = data['article'].str.replace('“','')
data['article'] = data['article'].str.replace(':','')
data['article'] = data['article'].str.replace('!','')
data['article'] = data['article'].str.replace('?','')
data['article'] = data['article'].str.replace('^','')
data['article'] = data['article'].str.replace('<','')
data.head()

,western_conservative,non_western,western_progressive,article
0,0.0,1.0,0.0,health authorities in one state have issued an...
1,0.0,1.0,0.0,kennedy saves the world podcast host kennedy a...
2,0.0,1.0,0.0,former counterterrorism analyst jonathan schan...
3,0.0,1.0,0.0,fox news flash top headlines are here check ou...
4,0.0,1.0,0.0,crowe is charged with harassment and stalking ...


In [9]:
# Removing stopwrods for word-embedding
from nltk.corpus import stopwords

def remove_stopwords(text):
    stop_words = set(stopwords.words('english'))
    words = text.split()
    filtered_words = [word for word in words if word.lower() not in stop_words]
    return ' '.join(filtered_words)

In [10]:
import nltk
nltk.download('stopwords')
data['article'] = data['article'].apply(remove_stopwords)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
data.head()

,western_conservative,non_western,western_progressive,article
0,0.0,1.0,0.0,health authorities one state issued urgent ale...
1,0.0,1.0,0.0,kennedy saves world podcast host kennedy fox n...
2,0.0,1.0,0.0,former counterterrorism analyst jonathan schan...
3,0.0,1.0,0.0,fox news flash top headlines check whats click...
4,0.0,1.0,0.0,crowe charged harassment stalking related acti...


In [12]:
# Saving cleaned data
data.to_csv('news_clean.csv',index=False)

***Word-Embedding***

In [13]:
data = pd.read_csv('datasets/news_clean.csv').dropna()
data.shape

(1162, 4)

In [14]:
!pip install gensim
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

# tokenizing text data
tokenized_data = [simple_preprocess(article) for article in data['article']]

# training Word2Vec model
# using recommended parameters
word2vec_model = Word2Vec(sentences=tokenized_data, vector_size=100, window=5, min_count=1, workers=4)


# retrieving word vectors for each token in the article
word_vectors = []
for tokens in tokenized_data:
    vectors = [word2vec_model.wv[token] for token in tokens if token in word2vec_model.wv]
    if vectors:
        article_vector = sum(vectors) / len(vectors)  # average the word vectors to get one vector per article
        word_vectors.append(article_vector)
    else:
        word_vectors.append(None)  # handle case where all tokens are out-of-vocabulary

# converting word_vectors to pandas series
word_vectors_series = pd.Series(word_vectors, name='word_embeddings')

# adding  the word vectors as a new column in your DataFrame
data['word_embeddings'] = word_vectors_series

In [15]:
data.head()

,western_conservative,non_western,western_progressive,article,word_embeddings
0,0.0,1.0,0.0,health authorities one state issued urgent ale...,"[-0.570655, 0.08599928, -0.1257813, -0.4425013..."
1,0.0,1.0,0.0,kennedy saves world podcast host kennedy fox n...,"[-0.87233, 0.10050296, -0.20962298, -0.6497976..."
2,0.0,1.0,0.0,former counterterrorism analyst jonathan schan...,"[-0.65388143, 0.13699661, -0.16996148, -0.6298..."
3,0.0,1.0,0.0,fox news flash top headlines check whats click...,"[-0.9647374, 0.13590194, -0.362687, -0.8459670..."
4,0.0,1.0,0.0,crowe charged harassment stalking related acti...,"[-0.67659926, 0.12015963, -0.14404652, -0.5377..."


In [16]:
# Right now the arrays are Series containing whole strings
# Converting to lists with floats:
import ast
import numpy as np

def convert_str(x):
    # Check if x is a string before attempting to convert
    if isinstance(x, str):
        neu = ast.literal_eval(x)
        return neu
    # If x is already a NumPy array, just return it
    elif isinstance(x, np.ndarray):
        return x.tolist() # convert to a normal Python list
    else:
        return x # For any other datatype

data = data.dropna(axis=0)

data['word_embeddings'] = data['word_embeddings'].apply(convert_str)

<ipython-input-16-fe7ee34fea23>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['word_embeddings'] = data['word_embeddings'].apply(convert_str)


In [17]:
type(data['word_embeddings'][0])
# List

list

In [18]:
type(data['word_embeddings'][0][0])
# Float

float

In [19]:
# Saving the word-embedded dataframe
data.to_csv('news_embedded.csv',index=False)

_____________________
**Model Fitting and Evaluation**

The problem is a Text Classification Problem.

For this kind of problem suitable models could be:

- Naive Bayes

- Support Vector Machines (SVM)

- Random Forest or Gradient Boosting Machines

- Neural Networks: deep learning models like Convolutional Neural Networks (CNNs) or Recurrent Neural Networks (RNNs).


However, since at the time of making this model the dataset has a limited number of samples, I will be focusing on the Naive Bayes model.

In [20]:
data = pd.read_csv('datasets/news_embedded.csv')

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score


# Concatenating the label columns into a single label column
data['label'] = data[['western_conservative', 'non_western', 'western_progressive']].idxmax(axis=1)

# Splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(data['article'], data['label'], test_size=0.2, random_state=42)

# Vectorizing the text data
vectorizer = CountVectorizer()
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

# Create the model and fitting it to the data
nb_classifier = MultinomialNB()
nb_classifier.fit(X_train_vectorized, y_train)

# Making predictions
y_pred = nb_classifier.predict(X_test_vectorized)

In [22]:
# Evaluating the model
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.9267241379310345


In [23]:
# Getting predicted probabilities
proba = nb_classifier.predict_proba(X_test_vectorized)

# Creating a DataFrame to display results
results_data = pd.DataFrame({'Article': X_test, 'Predicted Label': y_pred})
for i, label in enumerate(nb_classifier.classes_):
    results_data[label + ' Probability'] = proba[:, i]

# Printing the results
print(results_data.iloc[90])

Article                             former president donald trump seeking sweeping...
Predicted Label                                                   western_progressive
non_western Probability                                                           0.0
western_conservative Probability                                                  0.0
western_progressive Probability                                                   1.0
Name: 865, dtype: object


##SVM

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC # Importing SVC here as well

# Concatenating the label columns into a single label column
data['label'] = data[['western_conservative', 'non_western', 'western_progressive']].idxmax(axis=1)

# Splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(data['article'], data['label'], test_size=0.2, random_state=42)

# Vectorizing the text data
vectorizer = CountVectorizer()
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

# Create and train the SVM model
svm_classifier = SVC(kernel='linear', probability=True) # You can experiment with different kernels
svm_classifier.fit(X_train_vectorized, y_train)

# Make predictions
y_pred_svm = svm_classifier.predict(X_test_vectorized)

# Evaluate the model
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print("SVM Accuracy:", accuracy_svm)

# Getting predicted probabilities
proba_svm = svm_classifier.predict_proba(X_test_vectorized)

# Creating a DataFrame to display results
results_svm = pd.DataFrame({'Article': X_test, 'Predicted Label': y_pred_svm})
for i, label in enumerate(svm_classifier.classes_):
    results_svm[label + ' Probability'] = proba_svm[:, i]

# Printing the results
results_svm.iloc[90]

SVM Accuracy: 0.9827586206896551


,865
Article,former president donald trump seeking sweeping...
Predicted Label,western_progressive
non_western Probability,0.003484
western_conservative Probability,0.003528
western_progressive Probability,0.992988


##Random Forest

In [25]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split # Importing train_test_split

# Concatenating the label columns into a single label column
data['label'] = data[['western_conservative', 'non_western', 'western_progressive']].idxmax(axis=1)

# Splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(data['article'], data['label'], test_size=0.2, random_state=42)

# Vectorizing the text data
vectorizer = CountVectorizer()
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

# Create and train the Random Forest model
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42) # You can adjust n_estimators
rf_classifier.fit(X_train_vectorized, y_train)

# Make predictions
y_pred_rf = rf_classifier.predict(X_test_vectorized)

# Evaluate the model
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print("Random Forest Accuracy:", accuracy_rf)

# Getting predicted probabilities
proba_rf = rf_classifier.predict_proba(X_test_vectorized)

# Creating a DataFrame to display results
results_rf = pd.DataFrame({'Article': X_test, 'Predicted Label': y_pred_rf})
for i, label in enumerate(rf_classifier.classes_):
    results_rf[label + ' Probability'] = proba_rf[:, i]

# Printing the results
results_rf.iloc[90]

Random Forest Accuracy: 0.978448275862069


,865
Article,former president donald trump seeking sweeping...
Predicted Label,western_progressive
non_western Probability,0.01
western_conservative Probability,0.0
western_progressive Probability,0.99


## RNN

In [26]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Assuming 'data' DataFrame is already loaded and preprocessed as in the provided code.
# ... (Previous code for data loading and preprocessing) ...

# Prepare data for RNN
# Assuming you have a vocabulary size and maximum sequence length
vocab_size = 10000  # Example vocabulary size
max_len = 200      # Example maximum sequence length

# Convert text to sequences of integers (tokenization)
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(data['article'])
sequences = tokenizer.texts_to_sequences(data['article'])
padded_sequences = tf.keras.preprocessing.sequence.pad_sequences(sequences, maxlen=max_len, truncating='post', padding='post')

# One-hot encode the labels
labels = pd.get_dummies(data['label']).values


# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, labels, test_size=0.2, random_state=42)

# Define the RNN model
model = Sequential()
model.add(Embedding(vocab_size, 128, input_length=max_len)) # Embedding layer
model.add(LSTM(128)) # LSTM layer
model.add(Dense(3, activation='softmax')) # Output layer (3 classes)

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test))

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print("RNN Accuracy:", accuracy)

# Make predictions
predictions = model.predict(X_test)


Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


29/29 ━━━━━━━━━━━━━━━━━━━━ 21s 653ms/step - accuracy: 0.4414 - loss: 1.0582 - val_accuracy: 0.4655 - val_loss: 0.9880
Epoch 2/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 18s 633ms/step - accuracy: 0.5433 - loss: 0.9189 - val_accuracy: 0.6422 - val_loss: 0.8780
Epoch 3/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 13s 367ms/step - accuracy: 0.7873 - loss: 0.6137 - val_accuracy: 0.7629 - val_loss: 0.5461
Epoch 4/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 22s 405ms/step - accuracy: 0.9164 - loss: 0.2868 - val_accuracy: 0.7241 - val_loss: 0.6710
Epoch 5/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 19s 334ms/step - accuracy: 0.9471 - loss: 0.1764 - val_accuracy: 0.7284 - val_loss: 0.7181
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 87ms/step - accuracy: 0.7470 - loss: 0.6523
RNN Accuracy: 0.7284482717514038
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 186ms/step


In [27]:
# Get predicted probabilities
predictions = model.predict(X_test)

# Create a DataFrame to display results
results_rnn = pd.DataFrame({'Article': X_test[:, 0], 'Predicted Label': np.argmax(predictions, axis=1)}) # Assuming X_test contains tokenized sequences

# Get class labels from the one-hot encoded training labels
class_labels = list(pd.get_dummies(data['label']).columns)

for i, label in enumerate(class_labels):
    results_rnn[label + ' Probability'] = predictions[:, i]

# Printing the results for the 90th sample
results_rnn.iloc[90]


8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 86ms/step


,90
Article,36.000000
Predicted Label,2.000000
non_western Probability,0.000616
western_conservative Probability,0.000011
western_progressive Probability,0.999373
